In [1]:
import pandas as pd

# from data_processing.nkod_datasets import NKOD
import yaml

with open("config_experiments.yaml", "r") as f:
    config = yaml.safe_load(f)

# config = config
# dataset_portal = NKOD(config)
# dataset_portal.load()
# df = dataset_portal.super_df
df = pd.read_csv("../data/datove_sady.csv")

/tmp/ipykernel_423401/3810462410.py:13: DtypeWarning: Columns (13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/datove_sady.csv")


In [2]:
# get all unique themes
themes = df["téma"].explode().unique()
themes

array(['Hospodářství a finance', 'Vláda a veřejný sektor',
       'Regiony a města', 'Předběžné údaje', 'Zdraví',
       'Vzdělávání, kultura a sport', 'Věda a technika', 'Doprava',
       'Životní prostředí', 'Populace a společnost',
       'Spravedlnost, právní systém a veřejná bezpečnost',
       'úřední dokument', 'novinka', 'místní médium',
       'Zemědělství, rybolov, lesnictví a výživa', 'dopravní prostředek',
       'autobus', 'zdravotnická služba', 'sociální služby',
       'všeobecné školství', 'školství', 'střední vzdělání',
       'školský systém', 'veřejné školství', 'vodní tok', 'povodeň',
       'stupeň vzdělání', 'vzdělávání', 'umělecká výchova',
       'organizace dopravy', 'dopravní oblast', 'hromadné stravování',
       'školní vzdělávání', 'městská doprava', 'zdravotnická technologie',
       'hospitalizace', 'volný čas', 'ubytování studentů',
       'veřejná správa', 'předškolní výchova',
       'zdravotně postižený člověk', 'učitel', 'správní celek',
       '52 Ž

In [36]:
# get all keywords and their count
keywords = df["klíčová_slova"].dropna().explode()
keyword_counts = keywords.value_counts()
# store this to file
keyword_counts.to_csv("../data/keyword_counts.csv")
keyword_counts[keyword_counts <= 1]
print(keyword_counts)

klíčová_slova
prohlídky vozidel      18734
ÚPV                    16096
STK                    12678
SME                    12678
prohlídka vozidla      12676
                       ...  
bezpečnost ICT             1
elektronický prodej        1
AI                         1
cloudové služby            1
webové stránky             1
Name: count, Length: 5536, dtype: int64


In [39]:
# find kewyords that end in ","
keywords_wrong = keywords[keywords.str.endswith(",") | keywords.str.endswith(".")]
# remove that "." char from each word
keywords_wrong = keywords_wrong
keywords_wrong

46219        zákon č. 114/1992 Sb.
46261     vyhláška č. 453/2017 Sb.
188735             nahodilá těžba.
188736             nahodilá těžba.
188737             nahodilá těžba.
                    ...           
239234                106/1999 Sb.
239235                106/1999 Sb.
239236                106/1999 Sb.
239237                106/1999 Sb.
239238                106/1999 Sb.
Name: klíčová_slova, Length: 114, dtype: object

In [34]:
all_keywords = df["klíčová_slova"].dropna().explode().unique()
keywords_df = pd.DataFrame({
    'keyword': all_keywords,
    'keyword_lower': [k.lower() for k in all_keywords]
})

capitalization_variants = keywords_df.groupby('keyword_lower')['keyword'].apply(list)
capitalization_variants = capitalization_variants[capitalization_variants.apply(len) > 1]
capitalization_variants = capitalization_variants.reset_index().rename(columns={0: 'keywords'})
capitalization_variants['has_all_caps_keyword'] = capitalization_variants['keyword'].apply(
    lambda keywords: any(kw.isupper() and kw.isalpha() for kw in keywords)
)
capitalization_variants

,keyword_lower,keyword,has_all_caps_keyword
0,absolventi,"[Absolventi, absolventi]",False
1,adresy,"[Adresy, adresy]",False
2,agenda,"[agenda, Agenda]",False
3,akce,"[akce, Akce]",False
4,aktivní zóna,"[Aktivní zóna, aktivní zóna]",False
...,...,...,...
231,školská zařízení,"[Školská zařízení, školská zařízení]",False
232,školství,"[Školství, školství]",False
233,školy,"[Školy, školy]",False
234,železnice,"[Železnice, železnice]",False


In [22]:
from polyleven import levenshtein

words = df["klíčová_slova"].explode().unique()
words = [w.lower() for w in words if isinstance(w, str)]

sim = []
for i in range(len(words)):
    for j in range(i+1, len(words)):
        if levenshtein(words[i], words[j]) <= 1:
            sim.append((words[i], words[j]))
print(sim, sep="\n")
len(sim)

[('královéhradecký kraj', 'královéhradecký kraje'), ('královéhradecký kraj', 'ykrálovéhradecký kraj'), ('katalog otevřených dat', 'katalog otevřených dat'), ('hranice', 'hranice'), ('kraj', 'kraj'), ('kraj', 'kraje'), ('okresy', 'okres'), ('okresy', 'okres'), ('okresy', 'okresy'), ('nemocnice', 'nemocnice'), ('zdravotnictví', 'zdravotnictví'), ('inovační infrastruktury', 'inovační infrastruktura'), ('dotace', 'dotace'), ('voš', 'vo'), ('voš', 'vo'), ('voš', 'koš'), ('přijímací řízení', 'přijímací řízení'), ('informační centra', 'informační centra'), ('památky', 'památky'), ('památky', 'památka'), ('památky', 'památka'), ('hroby', 'hrob'), ('hroby', 'hroby'), ('organizace', 'organizace'), ('domovy mládeže', 'domov mládeže'), ('domovy mládeže', 'domovy mládeže'), ('internáty', 'internát'), ('internáty', 'internáty'), ('školství', 'školství'), ('obecní úřady', 'obecní úřad'), ('kontakty', 'kontakty'), ('kontakty', 'kontakt'), ('silnice', 'silnice'), ('opravy', 'oprava'), ('životní prostře

1928

In [ ]:
df_metadata = pd.read_csv("../data/datasets_with_metadata.csv")
df_metadata

In [ ]:
categories = df["kategorie_hvd_název"].explode().unique()
print(categories)
# count how many datasets have any category assigned - it is not none

count = df["kategorie_hvd_název"].notna().sum()
print(f"Number of datasets with category assigned: {count}")
# categories = df["kategorie_hvd_IRI"].explode().unique()
# print(categories)

# get df that has category assigned
df_with_category = df[df["kategorie_hvd_název"].notna()]
df_with_category